# Create Fact Table


### Reading Silver Data


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df_silver=spark.sql('select * from parquet.`abfss://silver@adlscarproject.dfs.core.windows.net/carSales`')


## Reading all DIMS tables


In [0]:
df_model = spark.sql("select * from cars_catalog.gold.dim_model")

In [0]:
df_branch = spark.sql("select * from cars_catalog.gold.dim_branch")

In [0]:
df_dealer = spark.sql("select * from cars_catalog.gold.dim_dealer")

In [0]:
df_date = spark.sql("select * from cars_catalog.gold.dim_date")

In [0]:
df_fact=df_silver.join(df_model, df_silver.Model_ID == df_model.Model_ID, 'left') \
                .join(df_branch, df_silver.Branch_ID == df_branch.Branch_ID, 'left') \
                .join(df_dealer, df_silver.Dealer_ID == df_dealer.Dealer_ID, 'left') \
                .join(df_date, df_silver.Date_ID == df_date.Date_ID, 'left') \
                .select(df_silver.Revenue,df_silver.Units_Sold,df_silver.Revenue_PerUnit,df_model.Model_Key,df_branch.Branch_Key,df_dealer.Dealer_Key,df_date.Date_Key)

Writing fact table

In [0]:
from delta.tables import *

In [0]:
try:
    table_exists = spark.catalog.tableExists('cars_catalog.gold.fact_sales')
except Exception:
    table_exists = False

if table_exists:
    delta_tbl = DeltaTable.forName(spark, 'cars_catalog.gold.fact_sales')
    delta_tbl.alias('trg').merge(df_fact.alias('src'), 'trg.Model_Key = src.Model_Key AND trg.Branch_Key = src.Branch_Key AND trg.Dealer_Key = src.Dealer_Key AND trg.Date_Key = src.Date_Key')\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    print('Merge completed successfully')


else:
    df_fact.write.format('delta')\
        .mode('Overwrite')\
        .option('path','abfss://gold@adlscarproject.dfs.core.windows.net/fact_sales')\
        .saveAsTable('cars_catalog.gold.fact_sales')   


In [0]:
%sql
select * from cars_catalog.gold.fact_sales